# Modelo Final: Deteccion Jerarquica de Clones de Codigo

## Objetivo
Este notebook clasifica pares de programas Python mediante un pipeline facil de explicar:

1. **Type I** se identifica con comparacion deterministica de codigo normalizado; no usa Random Forest.
2. **Type II** se decide con atributos de similitud lexica tipo **Baker** y un **Random Forest**.
3. **Type III / Type IV** se separan con un segundo **Random Forest** que combina Baker y analisis estructural **AST**; se conservan las extensiones de flujo, IR simbolica y complejidad del modelo actual.

El notebook parte de `Modelo_Actual_Capa1_Lexica.ipynb`: se reorganiza y documenta, no se sustituye el algoritmo por otra tecnologia.


## Organizacion del dataset

La carpeta utilizada es `DataBaseProject/pares_clones/` y contiene cuatro subcarpetas: `T1`, `T2`, `T3` y `T4`. Cada archivo `.py` almacena **dos snippets** separados por saltos de linea; esos dos fragmentos forman una instancia a clasificar.

| Carpeta | Clase del modelo | Interpretacion |
|---|---|---|
| `T1` | `type_I` | Copia igual salvo limpieza superficial, por ejemplo comentarios o espacios. |
| `T2` | `type_II` | Misma estructura lexica con cambios de nombres o literales. |
| `T3` | `type_III` | Modificaciones estructurales parciales: sentencias agregadas, removidas o reorganizadas. |
| `T4` | `type_IV` | Soluciones semanticamente relacionadas, pero con mayor diferencia de implementacion. |

El numero del archivo se recupera como `problem_id`. La division train/validacion/test se hace por ese identificador para impedir que variantes del mismo problema aparezcan tanto en entrenamiento como en evaluacion.


## Criterio de presentacion y relacion con trabajos anteriores

Se replica el patron observable en los notebooks `KMeans`, `Jerarquico`, `Codigo1`, `Codigo2`, `Codigo3` y `Codigo4`: cargar datos, mostrar distribucion, definir el algoritmo, entrenar, predecir y reportar metricas con tablas y graficas.

El AST se inspira en `analisis.py` y `arbol.py`: aquellos archivos construyen nodos y usan un `Visitor` para emitir LLVM IR (`.ll`). Aqui no se compila ni se ejecuta LLVM; el mismo principio de recorrido se usa para convertir cada programa Python en caracteristicas comparables para Random Forest.


## 1) Librerias y configuracion

**Que se hace:** importar herramientas estandar para datos, AST, Random Forest, metricas y graficas.

**Salida esperada:** rutas del dataset y semilla fija, necesarias para reproducir resultados.


In [ ]:
# Bloque: importacion de librerias y rutas reproducibles.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Librerias base
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any

import ast
import io
import json
import keyword
import random
import re
import tokenize
import warnings
from difflib import SequenceMatcher

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=SyntaxWarning)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

RUTA_BASE = Path.cwd()
RUTA_DATASET = RUTA_BASE / "DataBaseProject" if (RUTA_BASE / "DataBaseProject").exists() else RUTA_BASE
RUTA_PARES = RUTA_DATASET / "pares_clones"
print("SEED:", SEED)
print("RUTA_DATASET:", RUTA_DATASET)
print("RUTA_PARES:", RUTA_PARES)


In [ ]:
# Bloque: constantes, umbrales y variables Baker base.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Configuracion global del experimento
# Las carpetas T1-T4 son la etiqueta conocida de cada par en el dataset.
TIPOS_CLON = ["T1", "T2", "T3", "T4"]
TIPO_A_CLASE = {"T1": "type_I", "T2": "type_II", "T3": "type_III", "T4": "type_IV"}
TIPO_A_GRUPO = {"T1": "pares_t1", "T2": "pares_t2", "T3": "pares_t3", "T4": "pares_t4"}

PATRON_SEPARADOR_SNIPPETS = re.compile(r"\n\s*\n\s*\n+")
PATRON_ESPACIOS = re.compile(r"[ \t]+")
PATRON_SALTOS = re.compile(r"\n{3,}")
PATRON_FILE_ID = re.compile(r"_(\d+)\.py$")

# Type I exige igualdad; Type II utiliza probabilidad del RF2 para pasar a la siguiente capa.
UMBRAL_TIPO_I = 1.0
UMBRAL_PROB_TIPO_II = 0.50
MIN_MATCH_LEN_BAKER = 3
AST_VARIANT_OFICIAL = "reduced"
ESTRATEGIA_BALANCEO = "undersample"

ETIQUETAS_MODELO = ["type_I", "type_II", "type_III", "type_IV"]

# El RF de Type II solo observa las ocho variables Baker base.
BAKER_FEATURES_BASE = [
    "baker_match_total_ratio",
    "baker_match_max_ratio",
    "baker_num_blocks",
    "baker_sequence_ratio",
    "baker_edit_distance_norm",
    "baker_token_jaccard",
    "baker_common_token_coverage",
    "baker_len_diff_rel",
]


## 2) Reconstruccion de los pares

**Que se hace:** recorrer carpetas `T1` a `T4`, separar los dos snippets de cada archivo y construir `DatosPares`.

**Por que:** los clasificadores necesitan una tabla donde cada fila tenga `code_a`, `code_b` y la clase real.

**Salida esperada:** total de pares, problemas unicos y distribucion balanceada por clase.


In [ ]:
# Bloque: lectura de carpetas y reconstruccion del dataset.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Divide cada archivo en los dos programas que forman el par a comparar.
def separar_snippets(texto_archivo: str) -> list[str]:
    texto = texto_archivo.replace("\r\n", "\n").replace("\r", "\n").strip()
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not texto:
        return []
    return [p.strip() for p in PATRON_SEPARADOR_SNIPPETS.split(texto) if p.strip()]


# Funcion: Recorre T1, T2, T3 y T4 y reconstruye la tabla de pares etiquetados.
def cargar_pares_desde_carpetas(ruta_pares: Path) -> pd.DataFrame:
    filas = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for tipo in TIPOS_CLON:
        carpeta = ruta_pares / tipo
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if not carpeta.exists():
            raise FileNotFoundError(f"No existe carpeta esperada: {carpeta}")

        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for archivo in sorted(carpeta.glob("*.py")):
            contenido = archivo.read_text(encoding="utf-8", errors="replace")
            snippets = separar_snippets(contenido)
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if len(snippets) < 2:
                continue

            m = PATRON_FILE_ID.search(archivo.name)
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if not m:
                continue

            problem_id = int(m.group(1))
            filas.append(
                {
                    "is_clone": 1,
                    "clone_type": TIPO_A_CLASE[tipo],
                    "source_group": TIPO_A_GRUPO[tipo],
                    "filename": archivo.name,
                    "file_path": str(archivo.relative_to(RUTA_DATASET)).replace("/", "\\"),
                    "problem_id": problem_id,
                    "snippet_index_a": 0,
                    "snippet_index_b": 1,
                    "code_a": snippets[0],
                    "code_b": snippets[1],
                }
            )

    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not filas:
        raise RuntimeError("No se pudieron reconstruir pares desde carpetas.")

    return pd.DataFrame(filas)


DatosPares = cargar_pares_desde_carpetas(RUTA_PARES)
print("Filas reconstruidas:", len(DatosPares))
print("Problemas unicos:", DatosPares["problem_id"].nunique())
print("Distribucion por clase:")
print(DatosPares["clone_type"].value_counts().to_string())


## 3) Limpieza y regla deterministica Type I

**Que se hace:** eliminar comentarios, normalizar espacios, tokenizar y formar una firma canonica.

**Por que:** Type I representa copia practicamente exacta; compararla con una regla es mas transparente que entrenar un modelo para algo determinista.

**Salida esperada:** columnas limpias y firmas para cada lado del par.


In [ ]:
# Bloque: preprocesamiento lexico y firmas Type I.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Retira comentarios para que Type I compare codigo y no anotaciones.
def quitar_comentarios(codigo: str) -> str:
    lineas = codigo.expandtabs(4).splitlines()
    limpias = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for ln in lineas:
        limpias.append(ln.split("#", 1)[0])
    return "\n".join(limpias)


# Funcion: Uniforma espacios y saltos sin modificar la estructura esencial.
def normalizar_espacios(codigo: str) -> str:
    lineas = [PATRON_ESPACIOS.sub(" ", l).rstrip() for l in codigo.splitlines()]
    normalizado = "\n".join(lineas).strip()
    return PATRON_SALTOS.sub("\n\n", normalizado)


# Funcion: Aplica la limpieza comun usada antes de extraer atributos.
def preprocesar_codigo(codigo: str) -> str:
    return normalizar_espacios(quitar_comentarios(codigo))


# Funcion: Convierte codigo limpio en tokens simples para inspeccion.
def tokenizar_python_basico(codigo: str) -> list[str]:
    patron = r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|[][(){}.,:;+*/%=<>-]"
    return re.findall(patron, codigo)


# Funcion: Construye la firma exacta que decide Type I sin clasificador.
def firma_tipo_i_canonica(codigo: str) -> str:
    codigo_ok = preprocesar_codigo(codigo)
    return re.sub(r"\s+", "", codigo_ok)


DatosPares = DatosPares.copy()
DatosPares["code_a_clean"] = DatosPares["code_a"].map(preprocesar_codigo)
DatosPares["code_b_clean"] = DatosPares["code_b"].map(preprocesar_codigo)
DatosPares["tokens_a"] = DatosPares["code_a_clean"].map(tokenizar_python_basico)
DatosPares["tokens_b"] = DatosPares["code_b_clean"].map(tokenizar_python_basico)
DatosPares["token_text_a"] = DatosPares["tokens_a"].map(lambda t: " ".join(t))
DatosPares["token_text_b"] = DatosPares["tokens_b"].map(lambda t: " ".join(t))
DatosPares["type1_signature_a"] = DatosPares["code_a_clean"].map(firma_tipo_i_canonica)
DatosPares["type1_signature_b"] = DatosPares["code_b_clean"].map(firma_tipo_i_canonica)

print("Columnas disponibles:", len(DatosPares.columns))


## 4) Division de conjuntos por problema

**Que se hace:** dividir por `problem_id` en entrenamiento, validacion y prueba.

**Por que:** dividir filas al azar produciria fuga de informacion si el mismo ejercicio aparece en varios conjuntos.

**Salida esperada:** tabla de distribuciones por `split` y clase.


In [ ]:
# Bloque: split por grupo y balanceo controlado.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Separa train, validacion y test sin filtrar un problema entre conjuntos.
def split_por_grupo(
    df: pd.DataFrame,
    group_col: str,
    target_col: str,
    seed: int = 42,
    train_size: float = 0.7,
    val_size: float = 0.15,
    test_size: float = 0.15,
):
    proporcion_temp = val_size + test_size
    proporcion_test_rel = test_size / proporcion_temp

    gss_train = GroupShuffleSplit(n_splits=1, train_size=train_size, random_state=seed)
    idx_train_np, idx_temp_np = next(gss_train.split(df, y=df[target_col], groups=df[group_col]))

    df_temp = df.iloc[idx_temp_np]
    gss_temp = GroupShuffleSplit(n_splits=1, test_size=proporcion_test_rel, random_state=seed)
    idx_val_rel, idx_test_rel = next(gss_temp.split(df_temp, y=df_temp[target_col], groups=df_temp[group_col]))

    return df.index[idx_train_np], df_temp.index[idx_val_rel], df_temp.index[idx_test_rel]


# Funcion: Marca cada fila con el conjunto al que fue asignada.
def asignar_split(df: pd.DataFrame, idx_train, idx_val, idx_test, col_split: str = "split") -> pd.DataFrame:
    datos = df.copy()
    datos[col_split] = "unassigned"
    datos.loc[idx_train, col_split] = "train"
    datos.loc[idx_val, col_split] = "val"
    datos.loc[idx_test, col_split] = "test"
    return datos


# Funcion: Resume cantidad de pares, problemas y clases por particion.
def estadisticas_split(df: pd.DataFrame, split_col: str, target_col: str, group_col: str) -> list[dict[str, Any]]:
    resumen = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for nombre_split, df_split in df.groupby(split_col):
        conteos = df_split[target_col].value_counts().to_dict()
        resumen.append(
            {
                "split": nombre_split,
                "rows": int(len(df_split)),
                "unique_groups": int(df_split[group_col].nunique()),
                "class_distribution": {str(k): int(v) for k, v in conteos.items()},
            }
        )
    return resumen


# Funcion: Balancea solo entrenamiento cuando la estrategia lo requiere.
def balancear_train(df_train: pd.DataFrame, target_col: str, estrategia: str = "none", seed: int = 42):
    conteos = df_train[target_col].value_counts()
    info = {
        "strategy": estrategia,
        "rows_before": int(len(df_train)),
        "class_distribution_before": {str(k): int(v) for k, v in conteos.items()},
    }

    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if estrategia == "none" or len(conteos) <= 1:
        info["rows_after"] = int(len(df_train))
        info["class_distribution_after"] = info["class_distribution_before"]
        return df_train.copy(), info

    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if estrategia == "undersample":
        n_obj = int(conteos.min())
        rep = False
    # Evalua la alternativa siguiente cuando la condicion previa no se cumplio.
    elif estrategia == "oversample":
        n_obj = int(conteos.max())
        rep = True
    # Conserva el caso restante cuando ninguna condicion anterior aplica.
    else:
        info["rows_after"] = int(len(df_train))
        info["class_distribution_after"] = info["class_distribution_before"]
        return df_train.copy(), info

    partes = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for clase in conteos.index.tolist():
        df_clase = df_train[df_train[target_col] == clase]
        partes.append(df_clase.sample(n=n_obj, replace=rep, random_state=seed))

    out = pd.concat(partes, axis=0).sample(frac=1.0, random_state=seed).copy()
    c2 = out[target_col].value_counts()
    info["rows_after"] = int(len(out))
    info["class_distribution_after"] = {str(k): int(v) for k, v in c2.items()}
    return out, info


idx_train, idx_val, idx_test = split_por_grupo(
    df=DatosPares,
    group_col="problem_id",
    target_col="clone_type",
    seed=SEED + 100,
    train_size=0.7,
    val_size=0.15,
    test_size=0.15,
)

DatosModelo = asignar_split(DatosPares, idx_train, idx_val, idx_test)

print("Estadisticas split:")
# Recorre los elementos para acumular la salida correspondiente de este paso.
for fila in estadisticas_split(DatosModelo, "split", "clone_type", "problem_id"):
    print(fila)


## 5) Caracteristicas Baker para Type II

**Que se hace:** generalizar tokens (`ID`, `NUM`, `STR`) y medir coincidencias, distancia de edicion, LCS y Jaccard.

**Por que:** Type II cambia nombres o literales, pero conserva una forma lexica similar. El RF2 usa solo las ocho features Baker base para mantener esta etapa interpretable.

**Salida esperada:** funciones capaces de convertir cualquier tabla de pares en variables numericas Baker.


In [ ]:
# Bloque: extraccion de similitud Baker.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Calcula distancia de edicion entre secuencias de tokens.
def _levenshtein_tokens(seq_a: list[str], seq_b: list[str]) -> int:
    n, m = len(seq_a), len(seq_b)
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if n == 0:
        return m
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if m == 0:
        return n

    prev = list(range(m + 1))
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        ai = seq_a[i - 1]
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for j in range(1, m + 1):
            cost = 0 if ai == seq_b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return int(prev[m])


# Funcion: Calcula la subsecuencia comun mas larga entre tokens.
def _lcs_len_tokens(seq_a: list[str], seq_b: list[str]) -> int:
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not seq_a or not seq_b:
        return 0
    dp = [0] * (len(seq_b) + 1)
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for xa in seq_a:
        prev = 0
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for j, xb in enumerate(seq_b, start=1):
            temp = dp[j]
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if xa == xb:
                dp[j] = prev + 1
            # Conserva el caso restante cuando ninguna condicion anterior aplica.
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp
    return int(dp[-1])


# Funcion: Forma n-gramas de tokens generalizados para medir coincidencia local.
def _baker_ngram_set(tokens: list[str], n: int) -> set[tuple[str, ...]]:
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if len(tokens) < n:
        return set()
    return {tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1)}


# Funcion: Mide el traslape normalizado de dos conjuntos.
def _jaccard_set(a: set[Any], b: set[Any]) -> float:
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not a and not b:
        return 1.0
    union = a | b
    return float(len(a & b) / len(union)) if union else 0.0


# Funcion: Generaliza identificadores y literales siguiendo la idea Baker.
def baker_tokenizar_generalizar(codigo: str) -> list[str]:
    codigo_ok = codigo.replace("\r\n", "\n").replace("\r", "\n").expandtabs(4)
    ignorar = {
        tokenize.NL,
        tokenize.NEWLINE,
        tokenize.INDENT,
        tokenize.DEDENT,
        tokenize.COMMENT,
        tokenize.ENCODING,
        tokenize.ENDMARKER,
    }
    out = []
    # Intenta el analisis estructurado antes de usar un respaldo seguro.
    try:
        flujo = tokenize.generate_tokens(io.StringIO(codigo_ok).readline)
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for tok in flujo:
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if tok.type in ignorar:
                continue
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if tok.type == tokenize.NAME:
                out.append(tok.string if keyword.iskeyword(tok.string) else "ID")
            # Evalua la alternativa siguiente cuando la condicion previa no se cumplio.
            elif tok.type == tokenize.NUMBER:
                out.append("NUM")
            # Evalua la alternativa siguiente cuando la condicion previa no se cumplio.
            elif tok.type == tokenize.STRING:
                out.append("STR")
            # Evalua la alternativa siguiente cuando la condicion previa no se cumplio.
            elif tok.type == tokenize.OP:
                out.append(tok.string)
            # Conserva el caso restante cuando ninguna condicion anterior aplica.
            else:
                out.append(tok.string)
    # Mantiene el pipeline operativo si el fragmento no se puede interpretar completamente.
    except Exception:
        tokens = re.findall(r"[A-Za-z_]\w*|\d+|==|!=|<=|>=|[\[\](){}.,:;+*/%=<>-]", codigo_ok)
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for t in tokens:
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if re.fullmatch(r"[A-Za-z_]\w*", t):
                out.append(t if keyword.iskeyword(t) else "ID")
            # Evalua la alternativa siguiente cuando la condicion previa no se cumplio.
            elif re.fullmatch(r"\d+", t):
                out.append("NUM")
            # Conserva el caso restante cuando ninguna condicion anterior aplica.
            else:
                out.append(t)
    return out


# Funcion: Obtiene las similitudes lexicas Baker de un par de programas.
def baker_features_par(codigo_a: str, codigo_b: str, min_match_len: int = 3) -> dict[str, float]:
    ta = baker_tokenizar_generalizar(codigo_a)
    tb = baker_tokenizar_generalizar(codigo_b)

    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if len(ta) == 0 and len(tb) == 0:
        return {
            "baker_match_total_ratio": 1.0,
            "baker_match_max_ratio": 1.0,
            "baker_num_blocks": 1.0,
            "baker_sequence_ratio": 1.0,
            "baker_edit_distance_norm": 0.0,
            "baker_token_jaccard": 1.0,
            "baker_common_token_coverage": 1.0,
            "baker_len_diff_rel": 0.0,
            "baker_lcs_ratio": 1.0,
            "baker_bigram_jaccard": 1.0,
            "baker_trigram_jaccard": 1.0,
            "baker_keyword_overlap": 1.0,
            "baker_operator_overlap": 1.0,
            "baker_literal_density_diff": 0.0,
            "baker_identifier_density_diff": 0.0,
        }

    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if len(ta) == 0 or len(tb) == 0:
        return {
            "baker_match_total_ratio": 0.0,
            "baker_match_max_ratio": 0.0,
            "baker_num_blocks": 0.0,
            "baker_sequence_ratio": 0.0,
            "baker_edit_distance_norm": 1.0,
            "baker_token_jaccard": 0.0,
            "baker_common_token_coverage": 0.0,
            "baker_len_diff_rel": 1.0,
            "baker_lcs_ratio": 0.0,
            "baker_bigram_jaccard": 0.0,
            "baker_trigram_jaccard": 0.0,
            "baker_keyword_overlap": 0.0,
            "baker_operator_overlap": 0.0,
            "baker_literal_density_diff": 1.0,
            "baker_identifier_density_diff": 1.0,
        }

    matcher = SequenceMatcher(a=ta, b=tb, autojunk=False)
    blocks = [b for b in matcher.get_matching_blocks() if b.size >= min_match_len]
    total_match = float(sum(b.size for b in blocks))
    max_match = float(max([b.size for b in blocks], default=0.0))

    base_min = float(min(len(ta), len(tb)))
    base_max = float(max(len(ta), len(tb)))
    set_a, set_b = set(ta), set(tb)
    union = set_a | set_b
    inter = set_a & set_b

    edit_dist = float(_levenshtein_tokens(ta, tb))
    lcs_ratio = float(_lcs_len_tokens(ta, tb) / max(1.0, base_min))
    bigram_j = _jaccard_set(_baker_ngram_set(ta, 2), _baker_ngram_set(tb, 2))
    trigram_j = _jaccard_set(_baker_ngram_set(ta, 3), _baker_ngram_set(tb, 3))

    kw = set(keyword.kwlist)
    kw_a = {t for t in ta if t in kw}
    kw_b = {t for t in tb if t in kw}
    kw_overlap = _jaccard_set(kw_a, kw_b)

    op_pool = {
        "+",
        "-",
        "*",
        "/",
        "//",
        "%",
        "**",
        "=",
        "==",
        "!=",
        "<",
        ">",
        "<=",
        ">=",
        "+=",
        "-=",
        "*=",
        "/=",
        "%=",
        "and",
        "or",
        "not",
        "in",
        "is",
        "(",
        ")",
        "[",
        "]",
        "{",
        "}",
        ".",
        ",",
        ":",
    }
    op_a = {t for t in ta if t in op_pool}
    op_b = {t for t in tb if t in op_pool}
    op_overlap = _jaccard_set(op_a, op_b)

    lit_a = sum(1 for t in ta if t in {"NUM", "STR"}) / max(1.0, len(ta))
    lit_b = sum(1 for t in tb if t in {"NUM", "STR"}) / max(1.0, len(tb))
    id_a = sum(1 for t in ta if t == "ID") / max(1.0, len(ta))
    id_b = sum(1 for t in tb if t == "ID") / max(1.0, len(tb))

    return {
        "baker_match_total_ratio": total_match / base_min,
        "baker_match_max_ratio": max_match / base_min,
        "baker_num_blocks": float(len(blocks)),
        "baker_sequence_ratio": float(matcher.ratio()),
        "baker_edit_distance_norm": edit_dist / max(1.0, base_max),
        "baker_token_jaccard": float(len(inter) / len(union)) if len(union) > 0 else 0.0,
        "baker_common_token_coverage": float(len(inter) / max(1, min(len(set_a), len(set_b)))),
        "baker_len_diff_rel": abs(len(ta) - len(tb)) / max(1.0, base_max),
        "baker_lcs_ratio": lcs_ratio,
        "baker_bigram_jaccard": bigram_j,
        "baker_trigram_jaccard": trigram_j,
        "baker_keyword_overlap": kw_overlap,
        "baker_operator_overlap": op_overlap,
        "baker_literal_density_diff": abs(lit_a - lit_b),
        "baker_identifier_density_diff": abs(id_a - id_b),
    }


# Funcion: Construye la matriz Baker para todos los pares recibidos.
def construir_features_baker(df: pd.DataFrame, min_match_len: int = 3) -> pd.DataFrame:
    rows = [baker_features_par(a, b, min_match_len=min_match_len) for a, b in zip(df["code_a_clean"], df["code_b_clean"])]
    return pd.DataFrame(rows, index=df.index)


## 6) AST mediante Visitor

**Que se hace:** parsear cada snippet con `ast` de Python y recorrerlo con visitantes especializados.

**Por que:** igual que el Visitor de `arbol.py` permite generar LLVM en la practica anterior, aqui un Visitor separa el recorrido del arbol de las mediciones que se desean obtener.

**Salida esperada:** contadores estructurales, secuencias de flujo y una descripcion semantica ligera.


In [ ]:
# Bloque: visitors AST para estructura, flujo y semantica ligera.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Prepara fragmentos para que Python pueda formar un AST robustamente.
def _normalizar_codigo_para_ast(codigo: str) -> str:
    codigo = codigo.replace("\r\n", "\n").replace("\r", "\n").replace("\t", "    ").expandtabs(4)
    lineas = codigo.splitlines()
    out = []
    i = 0
    # Repite el proceso hasta consumir los elementos pendientes.
    while i < len(lineas):
        ln = lineas[i]
        s = ln.strip()
        indent_actual = len(ln) - len(ln.lstrip(" "))
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if re.match(r"^(else\s*:|elif\b.*:|except\b.*:|finally\s*:)$", s):
            ln = " " * indent_actual + "if True:"
            s = ln.strip()
        out.append(ln)
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if s and s.endswith(":"):
            j = i + 1
            # Repite el proceso hasta consumir los elementos pendientes.
            while j < len(lineas) and lineas[j].strip() == "":
                j += 1
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if j >= len(lineas):
                out.append(" " * (indent_actual + 4) + "pass")
            # Conserva el caso restante cuando ninguna condicion anterior aplica.
            else:
                indent_sig = len(lineas[j]) - len(lineas[j].lstrip(" "))
                # Atiende un caso especial o decide la rama apropiada del algoritmo.
                if indent_sig <= indent_actual:
                    out.append(" " * (indent_actual + 4) + "pass")
        i += 1
    texto = "\n".join(out).strip()
    return texto if texto else "pass"


# Funcion: Expresa una diferencia respecto al mayor valor observado.
def _diff_rel(a: float, b: float) -> float:
    den = max(1.0, abs(a), abs(b))
    return abs(a - b) / den


# Funcion: Calcula LCS para secuencias estructurales o semanticas.
def _lcs_len(seq_a: list[str], seq_b: list[str]) -> int:
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not seq_a or not seq_b:
        return 0
    dp = [0] * (len(seq_b) + 1)
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for xa in seq_a:
        prev = 0
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for j, xb in enumerate(seq_b, start=1):
            temp = dp[j]
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if xa == xb:
                dp[j] = prev + 1
            # Conserva el caso restante cuando ninguna condicion anterior aplica.
            else:
                dp[j] = max(dp[j], dp[j - 1])
            prev = temp
    return int(dp[-1])


# Funcion: Calcula ediciones entre secuencias AST o IR.
def _levenshtein_seq(seq_a: list[str], seq_b: list[str]) -> int:
    n, m = len(seq_a), len(seq_b)
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if n == 0:
        return m
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if m == 0:
        return n
    prev = list(range(m + 1))
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for i in range(1, n + 1):
        curr = [i] + [0] * m
        ai = seq_a[i - 1]
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for j in range(1, m + 1):
            cost = 0 if ai == seq_b[j - 1] else 1
            curr[j] = min(prev[j] + 1, curr[j - 1] + 1, prev[j - 1] + cost)
        prev = curr
    return int(prev[m])


# Funcion: Calcula similitud de orden entre dos secuencias.
def _seq_ratio(seq_a: list[str], seq_b: list[str]) -> float:
    return float(SequenceMatcher(a=seq_a, b=seq_b, autojunk=False).ratio())


# Funcion: Compara tipos de nodos presentes en dos AST.
def _jaccard_keys(mapa_a: dict[str, int], mapa_b: dict[str, int]) -> float:
    ka = set(mapa_a.keys())
    kb = set(mapa_b.keys())
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not ka and not kb:
        return 1.0
    union = ka | kb
    return float(len(ka & kb) / len(union)) if union else 0.0


# Funcion: Compara frecuencias de tipos de nodos AST.
def _weighted_overlap_counts(mapa_a: dict[str, int], mapa_b: dict[str, int]) -> float:
    keys = set(mapa_a.keys()) | set(mapa_b.keys())
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not keys:
        return 1.0
    inter = 0
    total = 0
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for k in keys:
        va = mapa_a.get(k, 0)
        vb = mapa_b.get(k, 0)
        inter += min(va, vb)
        total += max(va, vb)
    return float(inter / total) if total > 0 else 0.0


# Clase: Visitor que cuenta nodos y estructura del arbol sintactico.
class VisitorEstructuraAST(ast.NodeVisitor):
    # Funcion: Inicializa los acumuladores internos usados durante el recorrido AST.
    def __init__(self) -> None:
        self.total_nodes = 0
        self.max_depth = 0
        self.num_functions = 0
        self.num_loops = 0
        self.num_ifs = 0
        self.num_calls = 0
        self.num_imports = 0
        self.num_returns = 0
        self.num_assigns = 0
        self.num_comprehensions = 0
        self.num_try = 0
        self.num_branches = 0
        self.num_boolops = 0
        self.num_handlers = 0
        self.ids = set()
        self.type_counts: dict[str, int] = {}

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def visitar(self, root: ast.AST) -> None:
        self._recorrer(root, 0)

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _recorrer(self, node: ast.AST, depth: int) -> None:
        self.total_nodes += 1
        self.max_depth = max(self.max_depth, depth)
        t = type(node).__name__
        self.type_counts[t] = self.type_counts.get(t, 0) + 1

        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.Lambda)):
            self.num_functions += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.For, ast.AsyncFor, ast.While)):
            self.num_loops += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.If, ast.IfExp)):
            self.num_ifs += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Call):
            self.num_calls += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.Import, ast.ImportFrom)):
            self.num_imports += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Return):
            self.num_returns += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.Assign, ast.AnnAssign, ast.AugAssign)):
            self.num_assigns += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            self.num_comprehensions += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.Try, ast.TryStar)):
            self.num_try += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, (ast.If, ast.For, ast.AsyncFor, ast.While, ast.Try, ast.Match)):
            self.num_branches += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.BoolOp):
            self.num_boolops += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.ExceptHandler):
            self.num_handlers += 1
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Name):
            self.ids.add(node.id)
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.arg):
            self.ids.add(node.arg)

        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for ch in ast.iter_child_nodes(node):
            self._recorrer(ch, depth + 1)


# Clase: Visitor que registra caminos de control y llamadas.
class VisitorFlujoControl(ast.NodeVisitor):
    # Funcion: Inicializa los acumuladores internos usados durante el recorrido AST.
    def __init__(self) -> None:
        self.flow_sequence: list[str] = []
        self.call_sequence: list[str] = []
        self.branch_count = 0
        self.loop_count = 0
        self.return_count = 0
        self.call_count = 0
        self.try_except_count = 0
        self.max_control_nesting = 0
        self._control_nesting = 0

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _enter_control(self):
        self._control_nesting += 1
        self.max_control_nesting = max(self.max_control_nesting, self._control_nesting)

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _exit_control(self):
        self._control_nesting = max(0, self._control_nesting - 1)

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _call_name(self, func: ast.AST) -> str:
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(func, ast.Name):
            return func.id
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(func, ast.Attribute):
            base = self._call_name(func.value)
            return f"{base}.{func.attr}" if base else func.attr
        return ""

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_If(self, node: ast.If):
        self.flow_sequence.append("If")
        self.branch_count += 1
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_For(self, node: ast.For):
        self.flow_sequence.append("For")
        self.branch_count += 1
        self.loop_count += 1
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_AsyncFor(self, node: ast.AsyncFor):
        self.visit_For(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_While(self, node: ast.While):
        self.flow_sequence.append("While")
        self.branch_count += 1
        self.loop_count += 1
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Try(self, node: ast.Try):
        self.flow_sequence.append("Try")
        self.branch_count += 1 + len(node.handlers)
        self.try_except_count += len(node.handlers)
        self._enter_control()
        self.generic_visit(node)
        self._exit_control()

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_TryStar(self, node: ast.TryStar):
        self.visit_Try(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Return(self, node: ast.Return):
        self.flow_sequence.append("Return")
        self.return_count += 1
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Call(self, node: ast.Call):
        self.flow_sequence.append("Call")
        self.call_count += 1
        nombre = self._call_name(node.func)
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if nombre:
            self.call_sequence.append(nombre)
        self.generic_visit(node)


# Clase: Visitor que registra operaciones y sentencias normalizadas.
class VisitorSemanticaLigera(ast.NodeVisitor):
    # Funcion: Inicializa los acumuladores internos usados durante el recorrido AST.
    def __init__(self) -> None:
        self.op_sequence: list[str] = []
        self.stmt_sequence: list[str] = []
        self.ids = set()

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_BinOp(self, node: ast.BinOp):
        self.op_sequence.append("BIN_" + type(node.op).__name__)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_BoolOp(self, node: ast.BoolOp):
        self.op_sequence.append("BOOL_" + type(node.op).__name__)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_UnaryOp(self, node: ast.UnaryOp):
        self.op_sequence.append("UNARY_" + type(node.op).__name__)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Compare(self, node: ast.Compare):
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for op in node.ops:
            self.op_sequence.append("CMP_" + type(op).__name__)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Name(self, node: ast.Name):
        self.ids.add(node.id)

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def generic_visit(self, node: ast.AST):
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.stmt):
            self.stmt_sequence.append(type(node).__name__)
        super().generic_visit(node)


## 7) Vector final Type III / Type IV

**Que se hace:** combinar Baker con AST, flujo de control, una IR simbolica y complejidad.

**Aclaracion sobre IR:** la IR simbolica registra operaciones como llamadas, retornos o asignaciones; esta inspirada en la generacion `.ll`, pero no pretende ejecutar codigo ni sustituir LLVM.

**Salida esperada:** `construir_features_tipo_iii_iv` produce el vector enriquecido que alimenta al RF3.


In [ ]:
# Bloque: features AST, IR simbolica y complejidad.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
AST_METRIC_KEYS = [
    "ast_total_nodes",
    "ast_depth",
    "ast_num_functions",
    "ast_num_loops",
    "ast_num_ifs",
    "ast_num_calls",
    "ast_num_imports",
    "ast_num_returns",
    "ast_num_assigns",
    "ast_num_comprehensions",
    "ast_num_try",
    "ast_num_branches",
    "ast_cyclomatic",
    "ast_unique_identifiers",
]


# Funcion: Extrae conteos aproximados si un fragmento no puede parsearse.
def _extraer_fallback_ast(codigo_ok: str) -> dict[str, Any]:
    lineas = codigo_ok.splitlines()
    ids = {x for x in re.findall(r"[A-Za-z_]\w*", codigo_ok) if not keyword.iskeyword(x)}
    ramas = sum(1 for l in lineas if re.match(r"^\s*(if|for|while|try)\b", l))
    return {
        "ast_total_nodes": float(len(tokenizar_python_basico(codigo_ok))),
        "ast_depth": float(max((len(l) - len(l.lstrip(" "))) // 4 for l in lineas if l.strip()) if any(l.strip() for l in lineas) else 0),
        "ast_num_functions": float(sum(1 for l in lineas if re.match(r"^\s*(async\s+def|def)\s+", l))),
        "ast_num_loops": float(sum(1 for l in lineas if re.match(r"^\s*(for|while)\s+", l))),
        "ast_num_ifs": float(sum(1 for l in lineas if re.match(r"^\s*if\s+", l))),
        "ast_num_calls": float(len(re.findall(r"[A-Za-z_]\w*\s*\(", codigo_ok))),
        "ast_num_imports": float(sum(1 for l in lineas if re.match(r"^\s*(import|from)\s+", l))),
        "ast_num_returns": float(sum(1 for l in lineas if re.match(r"^\s*return\b", l))),
        "ast_num_assigns": float(sum(1 for l in lineas if re.search(r"[^=!<>]=[^=]", l))),
        "ast_num_comprehensions": float(len(re.findall(r"\[[^\]]+for\s+.+in\s+.+\]|\{[^\}]+for\s+.+in\s+.+\}", codigo_ok))),
        "ast_num_try": float(sum(1 for l in lineas if re.match(r"^\s*try\s*:", l))),
        "ast_num_branches": float(ramas),
        "ast_cyclomatic": float(1 + ramas),
        "ast_unique_identifiers": float(len(ids)),
        "ast_type_counts": {},
    }


# Funcion: Recorre el AST de un snippet y produce su perfil estructural.
def extraer_features_ast_snippet(codigo: str) -> dict[str, Any]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    # Intenta el analisis estructurado antes de usar un respaldo seguro.
    try:
        root = ast.parse(codigo_ok)
    # Mantiene el pipeline operativo si el fragmento no se puede interpretar completamente.
    except SyntaxError:
        return _extraer_fallback_ast(codigo_ok)

    v = VisitorEstructuraAST()
    v.visitar(root)
    cyclomatic = 1 + v.num_branches + v.num_boolops + v.num_handlers
    return {
        "ast_total_nodes": float(v.total_nodes),
        "ast_depth": float(v.max_depth),
        "ast_num_functions": float(v.num_functions),
        "ast_num_loops": float(v.num_loops),
        "ast_num_ifs": float(v.num_ifs),
        "ast_num_calls": float(v.num_calls),
        "ast_num_imports": float(v.num_imports),
        "ast_num_returns": float(v.num_returns),
        "ast_num_assigns": float(v.num_assigns),
        "ast_num_comprehensions": float(v.num_comprehensions),
        "ast_num_try": float(v.num_try),
        "ast_num_branches": float(v.num_branches),
        "ast_cyclomatic": float(cyclomatic),
        "ast_unique_identifiers": float(len(v.ids)),
        "ast_type_counts": v.type_counts,
    }


# Funcion: Extrae la secuencia de control y llamadas de un snippet.
def extraer_flujo_control_snippet(codigo: str) -> dict[str, Any]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    # Intenta el analisis estructurado antes de usar un respaldo seguro.
    try:
        root = ast.parse(codigo_ok)
    # Mantiene el pipeline operativo si el fragmento no se puede interpretar completamente.
    except SyntaxError:
        return {
            "flow_sequence": [],
            "call_sequence": [],
            "branch_count": 0.0,
            "loop_count": 0.0,
            "return_count": 0.0,
            "call_count": 0.0,
            "try_except_count": 0.0,
            "max_control_nesting": 0.0,
        }

    v = VisitorFlujoControl()
    v.visit(root)
    return {
        "flow_sequence": v.flow_sequence,
        "call_sequence": v.call_sequence,
        "branch_count": float(v.branch_count),
        "loop_count": float(v.loop_count),
        "return_count": float(v.return_count),
        "call_count": float(v.call_count),
        "try_except_count": float(v.try_except_count),
        "max_control_nesting": float(v.max_control_nesting),
    }


# Funcion: Obtiene secuencias AST que aproximan la semantica operativa.
def extraer_semantica_ligera_snippet(codigo: str) -> dict[str, Any]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    # Intenta el analisis estructurado antes de usar un respaldo seguro.
    try:
        root = ast.parse(codigo_ok)
    # Mantiene el pipeline operativo si el fragmento no se puede interpretar completamente.
    except SyntaxError:
        ops = re.findall(r"\+|\-|\*\*|\*|/|//|%|==|!=|<=|>=|<|>", codigo_ok)
        stmts = re.findall(r"^\s*([A-Za-z_]\w*)", codigo_ok, flags=re.M)
        ids = {x for x in re.findall(r"[A-Za-z_]\w*", codigo_ok) if not keyword.iskeyword(x)}
        return {"op_sequence": ops, "stmt_sequence": stmts, "ids": ids}

    v = VisitorSemanticaLigera()
    v.visit(root)
    return {"op_sequence": v.op_sequence, "stmt_sequence": v.stmt_sequence, "ids": v.ids}


# ------------------------------------------------------------
# IR semantico ligero (inspirado en flujo de compilador)
# ------------------------------------------------------------
# Clase: Visitor que forma una IR simbolica, sin emitir archivos LLVM.
class VisitorIRSemantico(ast.NodeVisitor):
    # Funcion: Inicializa los acumuladores internos usados durante el recorrido AST.
    def __init__(self) -> None:
        self.ir: list[str] = []
        self.var_map: dict[str, str] = {}
        self.next_var_id = 1

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _norm_var(self, nombre: str) -> str:
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if nombre not in self.var_map:
            self.var_map[nombre] = f'v{self.next_var_id}'
            self.next_var_id += 1
        return self.var_map[nombre]

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _expr_token(self, node: ast.AST | None) -> str:
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if node is None:
            return 'NONE'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Name):
            return f'VAR:{self._norm_var(node.id)}'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Constant):
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if isinstance(node.value, bool):
                return 'CONST:BOOL'
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if isinstance(node.value, (int, float, complex)):
                return 'CONST:NUM'
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if isinstance(node.value, str):
                return 'CONST:STR'
            return 'CONST:OTHER'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Call):
            return 'EXPR:CALL'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.BinOp):
            return f'EXPR:BIN_{type(node.op).__name__}'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.UnaryOp):
            return f'EXPR:UNARY_{type(node.op).__name__}'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.BoolOp):
            return f'EXPR:BOOL_{type(node.op).__name__}'
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(node, ast.Compare):
            return 'EXPR:COMPARE'
        return f'EXPR:{type(node).__name__}'

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _call_name(self, func: ast.AST) -> str:
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(func, ast.Name):
            return func.id.lower()
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if isinstance(func, ast.Attribute):
            return func.attr.lower()
        return 'call'

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_FunctionDef(self, node: ast.FunctionDef):
        self.ir.append('DEF_FUNC')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef):
        self.ir.append('DEF_FUNC')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Assign(self, node: ast.Assign):
        token_src = self._expr_token(node.value)
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for t in node.targets:
            # Atiende un caso especial o decide la rama apropiada del algoritmo.
            if isinstance(t, ast.Name):
                dst = self._norm_var(t.id)
                self.ir.append(f'ASSIGN:{dst}:{token_src}')
            # Conserva el caso restante cuando ninguna condicion anterior aplica.
            else:
                self.ir.append(f'ASSIGN:TARGET:{token_src}')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_AugAssign(self, node: ast.AugAssign):
        op = type(node.op).__name__
        lhs = self._expr_token(node.target)
        rhs = self._expr_token(node.value)
        self.ir.append(f'AUGASSIGN:{op}:{lhs}:{rhs}')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_If(self, node: ast.If):
        self.ir.append('BR_IF')
        self.ir.append('COND:' + self._expr_token(node.test))
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_For(self, node: ast.For):
        self.ir.append('LOOP_FOR')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_AsyncFor(self, node: ast.AsyncFor):
        self.ir.append('LOOP_FOR')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_While(self, node: ast.While):
        self.ir.append('LOOP_WHILE')
        self.ir.append('COND:' + self._expr_token(node.test))
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Try(self, node: ast.Try):
        self.ir.append('TRY_BLOCK')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_TryStar(self, node: ast.TryStar):
        self.ir.append('TRY_BLOCK')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Call(self, node: ast.Call):
        nombre = self._call_name(node.func)
        self.ir.append(f'CALL:{nombre}:{len(node.args)}')
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Return(self, node: ast.Return):
        self.ir.append('RETURN:' + self._expr_token(node.value))
        self.generic_visit(node)


# Funcion: Produce una IR simbolica corta inspirada en la practica LLVM.
def extraer_ir_semantico_snippet(codigo: str) -> list[str]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    # Intenta el analisis estructurado antes de usar un respaldo seguro.
    try:
        root = ast.parse(codigo_ok)
    # Mantiene el pipeline operativo si el fragmento no se puede interpretar completamente.
    except SyntaxError:
        # fallback muy simple si no parsea
        toks = re.findall(r'if|for|while|return|[A-Za-z_]\w*\s*\(|=', codigo_ok)
        return [f'RAW:{t.strip()}' for t in toks]

    v = VisitorIRSemantico()
    v.visit(root)
    return v.ir


# Funcion: Compara las dos secuencias IR simbolicas del par.
def _ir_features_par(codigo_a: str, codigo_b: str) -> dict[str, float]:
    ir_a = extraer_ir_semantico_snippet(codigo_a)
    ir_b = extraer_ir_semantico_snippet(codigo_b)

    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not ir_a and not ir_b:
        return {
            'ir_sequence_ratio': 1.0,
            'ir_lcs_similarity': 1.0,
            'ir_edit_distance_norm': 0.0,
            'ir_jaccard': 1.0,
            'ir_len_diff_rel': 0.0,
        }

    max_len = max(len(ir_a), len(ir_b))
    min_len = min(len(ir_a), len(ir_b))
    lcs = _lcs_len(ir_a, ir_b)
    edit = _levenshtein_seq(ir_a, ir_b)
    sa, sb = set(ir_a), set(ir_b)
    union = sa | sb
    jacc = 0.0 if not union else float(len(sa & sb) / len(union))

    return {
        'ir_sequence_ratio': _seq_ratio(ir_a, ir_b),
        'ir_lcs_similarity': (1.0 if max_len == 0 else 0.0) if min_len == 0 else float(lcs / min_len),
        'ir_edit_distance_norm': float(edit / max(1, max_len)),
        'ir_jaccard': jacc,
        'ir_len_diff_rel': abs(len(ir_a) - len(ir_b)) / max(1.0, float(max_len)),
    }


COMPLEXITY_METRIC_KEYS = [
    'complexity_cognitive',
    'complexity_halstead_volume',
    'complexity_operators_total',
    'complexity_operators_unique',
    'complexity_operands_total',
    'complexity_max_control_nesting',
    'complexity_writes',
    'complexity_calls',
]


# Clase: Visitor que resume complejidad del programa.
class VisitorComplejidadAST(ast.NodeVisitor):
    # Perfil de complejidad del AST, analogo a una pasada de analisis del compilador.
    # Funcion: Inicializa los acumuladores internos usados durante el recorrido AST.
    def __init__(self) -> None:
        self.operators: list[str] = []
        self.operands: list[str] = []
        self.cognitive = 0
        self.control_nesting = 0
        self.max_control_nesting = 0
        self.writes = 0
        self.calls = 0

    # Funcion: Encapsula una operacion auxiliar del pipeline para reutilizarla con cada par.
    def _visit_control(self, node: ast.AST, operador: str) -> None:
        self.operators.append(operador)
        self.cognitive += 1 + self.control_nesting
        self.control_nesting += 1
        self.max_control_nesting = max(self.max_control_nesting, self.control_nesting)
        self.generic_visit(node)
        self.control_nesting -= 1

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_If(self, node: ast.If):
        self._visit_control(node, 'IF')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_For(self, node: ast.For):
        self._visit_control(node, 'FOR')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_AsyncFor(self, node: ast.AsyncFor):
        self._visit_control(node, 'FOR')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_While(self, node: ast.While):
        self._visit_control(node, 'WHILE')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Try(self, node: ast.Try):
        self._visit_control(node, 'TRY')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_TryStar(self, node: ast.TryStar):
        self._visit_control(node, 'TRY')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_BinOp(self, node: ast.BinOp):
        self.operators.append('BIN_' + type(node.op).__name__)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_BoolOp(self, node: ast.BoolOp):
        self.operators.append('BOOL_' + type(node.op).__name__)
        self.cognitive += max(0, len(node.values) - 1)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_UnaryOp(self, node: ast.UnaryOp):
        self.operators.append('UNARY_' + type(node.op).__name__)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Compare(self, node: ast.Compare):
        self.operators.extend('CMP_' + type(op).__name__ for op in node.ops)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Assign(self, node: ast.Assign):
        self.operators.append('ASSIGN')
        self.writes += len(node.targets)
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_AugAssign(self, node: ast.AugAssign):
        self.operators.append('AUGASSIGN_' + type(node.op).__name__)
        self.writes += 1
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Call(self, node: ast.Call):
        self.operators.append('CALL')
        self.calls += 1
        self.generic_visit(node)

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Name(self, node: ast.Name):
        self.operands.append('ID')

    # Funcion: Visita este tipo de nodo AST y actualiza la representacion del programa.
    def visit_Constant(self, node: ast.Constant):
        tipo = 'BOOL' if isinstance(node.value, bool) else type(node.value).__name__.upper()
        self.operands.append('CONST_' + tipo)


# Funcion: Calcula complejidad cognitiva y volumen estructural de un AST.
def extraer_complejidad_snippet(codigo: str) -> dict[str, float]:
    codigo_ok = _normalizar_codigo_para_ast(codigo)
    # Intenta el analisis estructurado antes de usar un respaldo seguro.
    try:
        root = ast.parse(codigo_ok)
    # Mantiene el pipeline operativo si el fragmento no se puede interpretar completamente.
    except SyntaxError:
        return {k: 0.0 for k in COMPLEXITY_METRIC_KEYS}

    v = VisitorComplejidadAST()
    v.visit(root)
    longitud = len(v.operators) + len(v.operands)
    vocabulario = len(set(v.operators)) + len(set(v.operands))
    volumen = float(longitud * np.log2(max(1, vocabulario)))
    return {
        'complexity_cognitive': float(v.cognitive),
        'complexity_halstead_volume': volumen,
        'complexity_operators_total': float(len(v.operators)),
        'complexity_operators_unique': float(len(set(v.operators))),
        'complexity_operands_total': float(len(v.operands)),
        'complexity_max_control_nesting': float(v.max_control_nesting),
        'complexity_writes': float(v.writes),
        'complexity_calls': float(v.calls),
    }


# Funcion: Convierte diferencias de complejidad en columnas del par.
def construir_features_complejidad_par(df: pd.DataFrame) -> pd.DataFrame:
    fx_a = [extraer_complejidad_snippet(x) for x in df['code_a_clean']]
    fx_b = [extraer_complejidad_snippet(x) for x in df['code_b_clean']]
    filas = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for a, b in zip(fx_a, fx_b):
        fila = {}
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for k in COMPLEXITY_METRIC_KEYS:
            fila[f'{k}_diff_abs'] = abs(a[k] - b[k])
            fila[f'{k}_diff_rel'] = _diff_rel(a[k], b[k])
        filas.append(fila)
    return pd.DataFrame(filas, index=df.index)


# Funcion: Construye diferencias estructurales AST para cada par.
def construir_features_ast_par(df: pd.DataFrame) -> pd.DataFrame:
    fx_a = [extraer_features_ast_snippet(x) for x in df["code_a_clean"]]
    fx_b = [extraer_features_ast_snippet(x) for x in df["code_b_clean"]]
    filas = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for a, b in zip(fx_a, fx_b):
        fila = {}
        # Recorre los elementos para acumular la salida correspondiente de este paso.
        for k in AST_METRIC_KEYS:
            va = float(a[k])
            vb = float(b[k])
            fila[f"{k}_diff_abs"] = abs(va - vb)
            fila[f"{k}_diff_rel"] = _diff_rel(va, vb)
        fila["ast_type_jaccard_keys"] = _jaccard_keys(a["ast_type_counts"], b["ast_type_counts"])
        fila["ast_type_weighted_overlap"] = _weighted_overlap_counts(a["ast_type_counts"], b["ast_type_counts"])
        filas.append(fila)
    return pd.DataFrame(filas, index=df.index)


# Funcion: Conserva las variables AST definidas para el modelo oficial.
def seleccionar_ast_reducido(df_ast: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "ast_total_nodes_diff_rel",
        "ast_total_nodes_diff_abs",
        "ast_depth_diff_rel",
        "ast_depth_diff_abs",
        "ast_num_functions_diff_rel",
        "ast_num_functions_diff_abs",
        "ast_num_loops_diff_rel",
        "ast_num_loops_diff_abs",
        "ast_num_ifs_diff_rel",
        "ast_num_ifs_diff_abs",
        "ast_num_calls_diff_rel",
        "ast_num_calls_diff_abs",
        "ast_num_returns_diff_rel",
        "ast_num_returns_diff_abs",
        "ast_num_assigns_diff_rel",
        "ast_num_assigns_diff_abs",
        "ast_num_comprehensions_diff_rel",
        "ast_num_try_diff_rel",
        "ast_num_branches_diff_rel",
        "ast_num_branches_diff_abs",
        "ast_cyclomatic_diff_rel",
        "ast_cyclomatic_diff_abs",
        "ast_unique_identifiers_diff_rel",
        "ast_unique_identifiers_diff_abs",
        "ast_type_jaccard_keys",
        "ast_type_weighted_overlap",
    ]
    return df_ast[cols].copy()


# Funcion: Compara el orden relativo de llamadas a funciones.
def _call_seq_similarity(seq_a: list[str], seq_b: list[str]) -> float:
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not seq_a and not seq_b:
        return 1.0
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if not seq_a or not seq_b:
        return 0.0
    a = [x.split(".")[-1].lower() for x in seq_a]
    b = [x.split(".")[-1].lower() for x in seq_b]
    return _seq_ratio(a, b)


# Funcion: Compara ramas, ciclos, retornos y llamadas del par.
def construir_features_control_flow_par(df: pd.DataFrame) -> pd.DataFrame:
    fx_a = [extraer_flujo_control_snippet(x) for x in df["code_a_clean"]]
    fx_b = [extraer_flujo_control_snippet(x) for x in df["code_b_clean"]]
    filas = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for a, b in zip(fx_a, fx_b):
        seq_a = a["flow_sequence"]
        seq_b = b["flow_sequence"]
        lcs = _lcs_len(seq_a, seq_b)
        min_len = min(len(seq_a), len(seq_b))
        max_len = max(len(seq_a), len(seq_b))
        lcs_similarity = (1.0 if max_len == 0 else 0.0) if min_len == 0 else float(lcs / min_len)
        edit_norm = float(_levenshtein_seq(seq_a, seq_b) / max(1, max_len))
        filas.append(
            {
                "control_flow_sequence_ratio": _seq_ratio(seq_a, seq_b),
                "control_flow_lcs_similarity": lcs_similarity,
                "control_flow_edit_distance": edit_norm,
                "branch_count_diff_rel": _diff_rel(a["branch_count"], b["branch_count"]),
                "loop_count_diff_rel": _diff_rel(a["loop_count"], b["loop_count"]),
                "return_count_diff_rel": _diff_rel(a["return_count"], b["return_count"]),
                "call_count_diff_rel": _diff_rel(a["call_count"], b["call_count"]),
                "try_except_diff_rel": _diff_rel(a["try_except_count"], b["try_except_count"]),
                "max_control_nesting_diff_rel": _diff_rel(a["max_control_nesting"], b["max_control_nesting"]),
                "call_sequence_similarity": _call_seq_similarity(a["call_sequence"], b["call_sequence"]),
            }
        )
    return pd.DataFrame(filas, index=df.index)


# Funcion: Une secuencias AST enriquecidas e IR simbolica.
def construir_features_ast_enriquecido_par(df: pd.DataFrame) -> pd.DataFrame:
    filas = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for a, b in zip(df["code_a_clean"], df["code_b_clean"]):
        sx_a = extraer_semantica_ligera_snippet(a)
        sx_b = extraer_semantica_ligera_snippet(b)

        ops_a, ops_b = sx_a["op_sequence"], sx_b["op_sequence"]
        stmts_a, stmts_b = sx_a["stmt_sequence"], sx_b["stmt_sequence"]
        lcs_ops = _lcs_len(ops_a, ops_b)
        lcs_stmts = _lcs_len(stmts_a, stmts_b)
        min_ops = min(len(ops_a), len(ops_b))
        min_stmts = min(len(stmts_a), len(stmts_b))
        op_lcs = (1.0 if max(len(ops_a), len(ops_b)) == 0 else 0.0) if min_ops == 0 else float(lcs_ops / min_ops)
        stmt_lcs = (1.0 if max(len(stmts_a), len(stmts_b)) == 0 else 0.0) if min_stmts == 0 else float(lcs_stmts / min_stmts)
        ids_a, ids_b = sx_a["ids"], sx_b["ids"]
        union_ids = len(ids_a | ids_b)
        id_jaccard = 1.0 if union_ids == 0 else float(len(ids_a & ids_b) / union_ids)

        fx_ir = _ir_features_par(a, b)

        fila = {
            "ast_operator_sequence_ratio": _seq_ratio(ops_a, ops_b),
            "ast_operator_lcs_similarity": op_lcs,
            "ast_statement_sequence_ratio": _seq_ratio(stmts_a, stmts_b),
            "ast_statement_lcs_similarity": stmt_lcs,
            "ast_identifier_jaccard": id_jaccard,
        }
        fila.update(fx_ir)
        filas.append(fila)
    return pd.DataFrame(filas, index=df.index)


# Funcion: Selecciona exclusivamente features Baker para Type II.
def construir_features_tipo_ii(df: pd.DataFrame, min_match_len: int = 3) -> pd.DataFrame:
    fx = construir_features_baker(df, min_match_len=min_match_len)
    return fx[BAKER_FEATURES_BASE].copy()


# Funcion: Une Baker y AST enriquecido para separar Type III de Type IV.
def construir_features_tipo_iii_iv(df: pd.DataFrame, min_match_len: int = 3, ast_variant: str = "reduced") -> pd.DataFrame:
    fx_baker = construir_features_baker(df, min_match_len=min_match_len)
    fx_ast = construir_features_ast_par(df)
    fx_ast_sel = seleccionar_ast_reducido(fx_ast) if ast_variant == "reduced" else fx_ast
    fx_ast_extra = construir_features_ast_enriquecido_par(df)
    fx_cf = construir_features_control_flow_par(df)
    fx_complejidad = construir_features_complejidad_par(df)
    return pd.concat([fx_baker, fx_ast_sel, fx_ast_extra, fx_cf, fx_complejidad], axis=1)


## 8) Random Forest e hiperparametros

**Que se hace:** definir la regla de Type I, las metricas y dos Random Forest: RF2 para detectar Type II y RF3 para separar Type III de Type IV.

**Por que:** el pipeline respeta la dificultad creciente del problema: regla exacta, similitud lexica y finalmente similitud estructural.

| Hiperparametro | Funcion |
|---|---|
| `n_estimators` | Numero de arboles; mas arboles estabilizan la votacion. |
| `max_depth` | Limita profundidad para controlar sobreajuste. |
| `min_samples_leaf` | Exige ejemplos minimos en hojas. |
| `min_samples_split` | Exige ejemplos minimos para dividir un nodo. |
| `max_features` | Variables candidatas por division; `sqrt` diversifica arboles. |
| `class_weight` | Compensa diferencias de frecuencia de clases. |

La seleccion se realiza en validacion, no en test.


In [ ]:
# Bloque: regla Type I, metricas y seleccion de hiperparametros RF.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Decide si dos firmas corresponden a una copia exacta.
def es_tipo_i_deterministico(sig_a: str, sig_b: str, umbral: float = 1.0) -> bool:
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if sig_a == sig_b:
        return True
    return SequenceMatcher(None, sig_a, sig_b, autojunk=False).ratio() >= umbral


# Funcion: Aplica la regla Type I a una tabla completa de pares.
def detectar_tipo_i_deterministico(df: pd.DataFrame, umbral: float = 1.0) -> pd.Series:
    out = []
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for a, b in zip(df["type1_signature_a"], df["type1_signature_b"]):
        out.append(es_tipo_i_deterministico(a, b, umbral=umbral))
    return pd.Series(out, index=df.index, dtype=bool)


# Funcion: Calcula accuracy, precision, recall, F1 y matriz de confusion.
def evaluar_predicciones(y_true, y_pred, labels: list[str]) -> dict[str, Any]:
    acc = accuracy_score(y_true, y_pred)
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    reporte_dict = classification_report(y_true, y_pred, labels=labels, output_dict=True, zero_division=0)
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    return {
        "accuracy": float(acc),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f1_macro),
        "confusion_matrix": cm.tolist(),
        "classification_report_dict": reporte_dict,
    }


# Agrupa hiperparametros para mostrar y reutilizar la configuracion elegida.
@dataclass
class ConfigRF:
    n_estimators: int
    max_depth: int | None
    min_samples_leaf: int
    min_samples_split: int
    max_features: str | float | None
    class_weight: str = "balanced_subsample"


# Funcion: Instancia Random Forest usando una configuracion legible.
def _crear_rf(cfg: ConfigRF, random_state: int) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=cfg.n_estimators,
        max_depth=cfg.max_depth,
        min_samples_leaf=cfg.min_samples_leaf,
        min_samples_split=cfg.min_samples_split,
        max_features=cfg.max_features,
        class_weight=cfg.class_weight,
        random_state=random_state,
        n_jobs=-1,
    )


# Funcion: Declara hiperparametros candidatos para la etapa Type II.
def _grilla_rf_tipo2() -> list[ConfigRF]:
    return [
        ConfigRF(500, None, 1, 2, "sqrt"),
        ConfigRF(700, None, 1, 2, "sqrt"),
        ConfigRF(700, 25, 1, 2, "sqrt"),
        ConfigRF(900, None, 2, 5, "sqrt"),
    ]


# Funcion: Declara hiperparametros candidatos para la etapa Type III/IV.
def _grilla_rf_tipo34() -> list[ConfigRF]:
    return [
        ConfigRF(600, None, 1, 2, "sqrt"),
        ConfigRF(800, None, 1, 2, "sqrt"),
        ConfigRF(900, None, 2, 2, "sqrt"),
        ConfigRF(900, 30, 1, 5, "sqrt"),
    ]


# Funcion: Elige RF2 con F1 binario de validacion.
def seleccionar_mejor_rf_tipo2(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series, seed: int):
    mejor_modelo, mejor_cfg, mejor_f1 = None, None, -1.0
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for cfg in _grilla_rf_tipo2():
        modelo = _crear_rf(cfg, random_state=seed)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
        score = f1_score(y_val, pred, average="binary", zero_division=0)
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if score > mejor_f1:
            mejor_modelo, mejor_cfg, mejor_f1 = modelo, cfg, score
    return mejor_modelo, mejor_cfg, float(mejor_f1)


# Funcion: Elige RF3 con F1 macro de validacion.
def seleccionar_mejor_rf_tipo34(X_train: pd.DataFrame, y_train: pd.Series, X_val: pd.DataFrame, y_val: pd.Series, seed: int):
    mejor_modelo, mejor_cfg, mejor_f1 = None, None, -1.0
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for cfg in _grilla_rf_tipo34():
        modelo = _crear_rf(cfg, random_state=seed)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
        score = f1_score(y_val, pred, average="macro", zero_division=0)
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if score > mejor_f1:
            mejor_modelo, mejor_cfg, mejor_f1 = modelo, cfg, score
    return mejor_modelo, mejor_cfg, float(mejor_f1)


## 9) Entrenamiento del pipeline jerarquico final

**Que se hace:** entrenar RF2 y RF3, luego predecir en el orden Type I -> Type II -> Type III/IV.

**Salida esperada:** modelos entrenados, configuraciones elegidas, predicciones, importancias y metricas.


In [ ]:
# Bloque: entrenamiento y prediccion jerarquica.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Ejecuta en orden Type I, Type II y finalmente Type III/IV.
def predecir_jerarquico(
    df: pd.DataFrame,
    modelo_tipo_ii: RandomForestClassifier,
    modelo_tipo_iii_iv: RandomForestClassifier,
    min_match_len: int = 3,
    ast_variant: str = "reduced",
    umbral_tipo_i: float = 1.0,
    umbral_prob_tipo_ii: float = 0.5,
):
    pred = pd.Series(index=df.index, dtype="object")

    # Capa 1: Type I deterministico
    mask_i = detectar_tipo_i_deterministico(df, umbral=umbral_tipo_i)
    pred.loc[mask_i] = "type_I"

    # Capa 2: RF Type II
    idx_restante = df.index[~mask_i]
    # Atiende un caso especial o decide la rama apropiada del algoritmo.
    if len(idx_restante) > 0:
        X_t2 = construir_features_tipo_ii(df.loc[idx_restante], min_match_len=min_match_len)
        prob_t2 = modelo_tipo_ii.predict_proba(X_t2)[:, 1]
        pred_t2_bin = (prob_t2 >= umbral_prob_tipo_ii).astype(int)
        idx_type_ii = idx_restante[pred_t2_bin == 1]
        pred.loc[idx_type_ii] = "type_II"

        # Capa 3: RF Type III/IV
        idx_restante_2 = pred.index[pred.isna()]
        # Atiende un caso especial o decide la rama apropiada del algoritmo.
        if len(idx_restante_2) > 0:
            X_t34 = construir_features_tipo_iii_iv(
                df.loc[idx_restante_2],
                min_match_len=min_match_len,
                ast_variant=ast_variant,
            )
            pred.loc[idx_restante_2] = modelo_tipo_iii_iv.predict(X_t34)

    pred = pred.fillna("type_IV")
    resumen = {
        "pred_type_I": int((pred == "type_I").sum()),
        "pred_type_II": int((pred == "type_II").sum()),
        "pred_type_III": int((pred == "type_III").sum()),
        "pred_type_IV": int((pred == "type_IV").sum()),
    }
    return pred, resumen


# Funcion: Entrena las etapas y devuelve modelos, metricas e importancias.
def entrenar_evaluar_modelo_jerarquico(
    datos_task: pd.DataFrame,
    columna_target: str,
    etiquetas: list[str],
    seed: int,
    estrategia_balanceo: str = "none",
    min_match_len: int = 3,
    ast_variant: str = "reduced",
    umbral_tipo_i: float = 1.0,
    umbral_prob_tipo_ii: float = 0.5,
):
    train_raw = datos_task[datos_task["split"] == "train"].copy()
    val = datos_task[datos_task["split"] == "val"].copy()
    test = datos_task[datos_task["split"] == "test"].copy()

    train_balanceado, info_balanceo = balancear_train(train_raw, columna_target, estrategia_balanceo, seed)

    mask_i_train = detectar_tipo_i_deterministico(train_balanceado, umbral=umbral_tipo_i)
    train_etapa2 = train_balanceado.loc[~mask_i_train].copy()

    y_train_t2 = (train_etapa2[columna_target] == "type_II").astype(int)
    X_train_t2 = construir_features_tipo_ii(train_etapa2, min_match_len=min_match_len)

    val_no_i_mask = ~detectar_tipo_i_deterministico(val, umbral=umbral_tipo_i)
    val_t2 = val.loc[val_no_i_mask]
    y_val_t2 = (val_t2[columna_target] == "type_II").astype(int)
    X_val_t2 = construir_features_tipo_ii(val_t2, min_match_len=min_match_len)

    modelo_tipo_ii, cfg_t2, f1_t2_val = seleccionar_mejor_rf_tipo2(X_train_t2, y_train_t2, X_val_t2, y_val_t2, seed=seed)

    train_etapa3 = train_etapa2[train_etapa2[columna_target].isin(["type_III", "type_IV"])].copy()
    y_train_t34 = train_etapa3[columna_target]
    X_train_t34 = construir_features_tipo_iii_iv(train_etapa3, min_match_len=min_match_len, ast_variant=ast_variant)

    val_t34 = val_t2[val_t2[columna_target].isin(["type_III", "type_IV"])].copy()
    y_val_t34 = val_t34[columna_target]
    X_val_t34 = construir_features_tipo_iii_iv(val_t34, min_match_len=min_match_len, ast_variant=ast_variant)

    modelo_tipo_iii_iv, cfg_t34, f1_t34_val = seleccionar_mejor_rf_tipo34(X_train_t34, y_train_t34, X_val_t34, y_val_t34, seed=seed + 1030)

    pred_val, resumen_val = predecir_jerarquico(
        val,
        modelo_tipo_ii=modelo_tipo_ii,
        modelo_tipo_iii_iv=modelo_tipo_iii_iv,
        min_match_len=min_match_len,
        ast_variant=ast_variant,
        umbral_tipo_i=umbral_tipo_i,
        umbral_prob_tipo_ii=umbral_prob_tipo_ii,
    )

    pred_test, resumen_test = predecir_jerarquico(
        test,
        modelo_tipo_ii=modelo_tipo_ii,
        modelo_tipo_iii_iv=modelo_tipo_iii_iv,
        min_match_len=min_match_len,
        ast_variant=ast_variant,
        umbral_tipo_i=umbral_tipo_i,
        umbral_prob_tipo_ii=umbral_prob_tipo_ii,
    )

    metricas_val = evaluar_predicciones(val[columna_target], pred_val, labels=etiquetas)
    metricas_test = evaluar_predicciones(test[columna_target], pred_test, labels=etiquetas)

    importancia_t2 = pd.DataFrame({"feature": X_train_t2.columns, "importance": modelo_tipo_ii.feature_importances_}).sort_values("importance", ascending=False)
    importancia_t34 = pd.DataFrame({"feature": X_train_t34.columns, "importance": modelo_tipo_iii_iv.feature_importances_}).sort_values("importance", ascending=False)

    return {
        "info_balanceo": info_balanceo,
        "metricas_val": metricas_val,
        "metricas_test": metricas_test,
        "pred_val": pred_val,
        "pred_test": pred_test,
        "modelo_tipo_ii": modelo_tipo_ii,
        "modelo_tipo_iii_iv": modelo_tipo_iii_iv,
        "num_features_tipo_ii": int(X_train_t2.shape[1]),
        "num_features_tipo_iii_iv": int(X_train_t34.shape[1]),
        "feature_importance_tipo_ii": importancia_t2,
        "feature_importance_tipo_iii_iv": importancia_t34,
        "resumen_pred_val": resumen_val,
        "resumen_pred_test": resumen_test,
        "umbral_tipo_i": float(umbral_tipo_i),
        "umbral_prob_tipo_ii": float(umbral_prob_tipo_ii),
        "cfg_tipo_ii": cfg_t2,
        "cfg_tipo_iii_iv": cfg_t34,
        "f1_val_tipo_ii": f1_t2_val,
        "f1_val_tipo_iii_iv": f1_t34_val,
    }


ResultadoModelo = entrenar_evaluar_modelo_jerarquico(
    datos_task=DatosModelo,
    columna_target="clone_type",
    etiquetas=ETIQUETAS_MODELO,
    seed=SEED + 100,
    estrategia_balanceo=ESTRATEGIA_BALANCEO,
    min_match_len=MIN_MATCH_LEN_BAKER,
    ast_variant=AST_VARIANT_OFICIAL,
    umbral_tipo_i=UMBRAL_TIPO_I,
    umbral_prob_tipo_ii=UMBRAL_PROB_TIPO_II,
)

print("Modelo listo.")
print("Features tipo_II:", ResultadoModelo["num_features_tipo_ii"])
print("Features tipo_III/type_IV:", ResultadoModelo["num_features_tipo_iii_iv"])
print("Mejor F1 val tipo_II:", round(ResultadoModelo["f1_val_tipo_ii"], 6))
print("Mejor F1 val tipo_III/type_IV:", round(ResultadoModelo["f1_val_tipo_iii_iv"], 6))


## 10) Evaluacion final

**Que se hace:** presentar `accuracy`, `precision`, `recall`, `F1-score`, `F1 macro` y la matriz de confusion, siguiendo el formato de los ejercicios de metricas.

**Salida esperada:** desempeno en validacion y test, ademas del ranking inicial de variables del RF.


In [ ]:
# Bloque: reporte de clasificacion y matriz de confusion.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
metricas_val = ResultadoModelo["metricas_val"]
metricas_test = ResultadoModelo["metricas_test"]

print("=== VALIDACION ===")
print("accuracy:", round(metricas_val["accuracy"], 6))
print("f1_macro:", round(metricas_val["f1_macro"], 6))

print("\n=== TEST ===")
print("accuracy:", round(metricas_test["accuracy"], 6))
print("f1_macro:", round(metricas_test["f1_macro"], 6))

reporte_test = pd.DataFrame(metricas_test["classification_report_dict"]).T
display(reporte_test)

cm = np.array(metricas_test["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=ETIQUETAS_MODELO).plot(cmap="Blues", ax=ax, colorbar=False)
ax.set_title("Matriz de confusion (TEST)")
plt.tight_layout()
plt.show()

print("\nTop 15 features RF tipo_II:")
display(ResultadoModelo["feature_importance_tipo_ii"].head(15))
print("Top 20 features RF tipo_III/type_IV:")
display(ResultadoModelo["feature_importance_tipo_iii_iv"].head(20))


## 11) Experimento A: baseline Baker contra modelo final

**Pregunta:** para separar Type III de Type IV, se evalua si agregar AST/IR/complejidad aporta frente a usar solo Baker.

Se conserva la regla Type I y el RF2. El baseline reemplaza solamente RF3 por un Random Forest entrenado con Baker, de modo que la comparacion sea entendible.


In [ ]:

# Experimento: comparar el vector Baker solo contra el vector final en la etapa dificil.
# La comparacion conserva Type I y RF2; solamente cambia el clasificador Type III/IV.
def predecir_con_baseline_t34(df: pd.DataFrame, AlgoritmoBaselineT34: RandomForestClassifier) -> pd.Series:
    # Crea una serie vacia donde se asignara la clase en orden jerarquico.
    Hipotesis = pd.Series(index=df.index, dtype="object")
    # La primera etapa mantiene la regla deterministica del modelo final.
    MascaraTipoI = detectar_tipo_i_deterministico(df, umbral=UMBRAL_TIPO_I)
    Hipotesis.loc[MascaraTipoI] = "type_I"
    # El RF2 es exactamente el mismo modelo final, para aislar el efecto de T34.
    IndiceNoTipoI = df.index[~MascaraTipoI]
    if len(IndiceNoTipoI) > 0:
        CaracteristicasTipoII = construir_features_tipo_ii(df.loc[IndiceNoTipoI], MIN_MATCH_LEN_BAKER)
        ProbabilidadTipoII = ResultadoModelo["modelo_tipo_ii"].predict_proba(CaracteristicasTipoII)[:, 1]
        MascaraTipoII = ProbabilidadTipoII >= UMBRAL_PROB_TIPO_II
        Hipotesis.loc[IndiceNoTipoI[MascaraTipoII]] = "type_II"
        # A los pares restantes se les aplica el baseline Baker sin AST.
        IndiceT34 = Hipotesis.index[Hipotesis.isna()]
        if len(IndiceT34) > 0:
            CaracteristicasBaseline = construir_features_baker(df.loc[IndiceT34], MIN_MATCH_LEN_BAKER)
            Hipotesis.loc[IndiceT34] = AlgoritmoBaselineT34.predict(CaracteristicasBaseline)
    # Este respaldo evita valores vacios en caso de un conjunto excepcional.
    return Hipotesis.fillna("type_IV")


# Se entrenan Type III y Type IV del baseline usando las mismas particiones.
DatosTrainT34 = DatosModelo[(DatosModelo["split"] == "train") & DatosModelo["clone_type"].isin(["type_III", "type_IV"])].copy()
ClasesTrainT34 = DatosTrainT34["clone_type"]
CaracteristicasTrainBaseline = construir_features_baker(DatosTrainT34, MIN_MATCH_LEN_BAKER)
# Se reutilizan los hiperparametros elegidos para que la diferencia principal sea el conjunto de features.
AlgoritmoBaselineT34 = _crear_rf(ResultadoModelo["cfg_tipo_iii_iv"], random_state=SEED + 1130)
AlgoritmoBaselineT34.fit(CaracteristicasTrainBaseline, ClasesTrainT34)

# Se evalua el pipeline baseline completo sobre el mismo conjunto test reservado.
DatosTest = DatosModelo[DatosModelo["split"] == "test"].copy()
HipotesisBaselineTest = predecir_con_baseline_t34(DatosTest, AlgoritmoBaselineT34)
MetricasBaselineTest = evaluar_predicciones(DatosTest["clone_type"], HipotesisBaselineTest, ETIQUETAS_MODELO)

# Se muestran ambos resultados en una tabla sencilla, como en los ejercicios de metricas.
TablaComparacionModelos = pd.DataFrame([
    {
        "Modelo": "Baseline: Baker en T34",
        "Num_features_T34": int(CaracteristicasTrainBaseline.shape[1]),
        "Accuracy_test": MetricasBaselineTest["accuracy"],
        "Precision_macro_test": MetricasBaselineTest["precision_macro"],
        "Recall_macro_test": MetricasBaselineTest["recall_macro"],
        "F1_macro_test": MetricasBaselineTest["f1_macro"],
    },
    {
        "Modelo": "Final: Baker + AST/IR/complejidad",
        "Num_features_T34": ResultadoModelo["num_features_tipo_iii_iv"],
        "Accuracy_test": ResultadoModelo["metricas_test"]["accuracy"],
        "Precision_macro_test": ResultadoModelo["metricas_test"]["precision_macro"],
        "Recall_macro_test": ResultadoModelo["metricas_test"]["recall_macro"],
        "F1_macro_test": ResultadoModelo["metricas_test"]["f1_macro"],
    },
])
print("Comparacion baseline contra modelo final:")
display(TablaComparacionModelos.round(6))


## 12) Experimento B: movimiento del umbral de Type II

**Pregunta:** se mide que pasa si RF2 exige mayor o menor probabilidad antes de declarar `type_II`.

Se usa validacion para no tomar decisiones mirando test. El umbral Type I queda fijo en `1.0`, porque disminuirlo cambiaria la definicion de copia exacta.


In [ ]:

# Experimento: mover el umbral probabilistico de Type II usando solo validacion.
# El umbral Type I permanece en 1.0 porque define copia deterministica exacta.
DatosVal = DatosModelo[DatosModelo["split"] == "val"].copy()
MascaraTipoIVal = detectar_tipo_i_deterministico(DatosVal, umbral=UMBRAL_TIPO_I)
IndiceRFVal = DatosVal.index[~MascaraTipoIVal]
# Las features se calculan una vez para que probar umbrales no repita trabajo costoso.
CaracteristicasTipoIIVal = construir_features_tipo_ii(DatosVal.loc[IndiceRFVal], MIN_MATCH_LEN_BAKER)
ProbabilidadTipoIIVal = ResultadoModelo["modelo_tipo_ii"].predict_proba(CaracteristicasTipoIIVal)[:, 1]
CaracteristicasT34Val = construir_features_tipo_iii_iv(DatosVal.loc[IndiceRFVal], MIN_MATCH_LEN_BAKER, AST_VARIANT_OFICIAL)
HipotesisT34Val = pd.Series(ResultadoModelo["modelo_tipo_iii_iv"].predict(CaracteristicasT34Val), index=IndiceRFVal)

ResultadosUmbral = []
# Cada umbral altera cuantos pares acepta RF2 como Type II antes de pasar a RF3.
for UmbralTipoII in [0.35, 0.45, 0.50, 0.55, 0.65]:
    HipotesisVal = pd.Series(index=DatosVal.index, dtype="object")
    HipotesisVal.loc[MascaraTipoIVal] = "type_I"
    MascaraAceptadaT2 = ProbabilidadTipoIIVal >= UmbralTipoII
    HipotesisVal.loc[IndiceRFVal[MascaraAceptadaT2]] = "type_II"
    IndicePasaT34 = IndiceRFVal[~MascaraAceptadaT2]
    HipotesisVal.loc[IndicePasaT34] = HipotesisT34Val.loc[IndicePasaT34]
    MetricasUmbral = evaluar_predicciones(DatosVal["clone_type"], HipotesisVal, ETIQUETAS_MODELO)
    ResultadosUmbral.append({
        "Umbral_RF_Tipo_II": UmbralTipoII,
        "Accuracy_val": MetricasUmbral["accuracy"],
        "Precision_macro_val": MetricasUmbral["precision_macro"],
        "Recall_macro_val": MetricasUmbral["recall_macro"],
        "F1_macro_val": MetricasUmbral["f1_macro"],
    })

TablaUmbrales = pd.DataFrame(ResultadosUmbral)
print("Movimiento del umbral RF Type II en validacion:")
display(TablaUmbrales.round(6))
plt.figure(figsize=(8, 4))
plt.plot(TablaUmbrales["Umbral_RF_Tipo_II"], TablaUmbrales["F1_macro_val"], marker="o", color="green")
plt.title("F1 macro de validacion al mover el umbral de Type II")
plt.xlabel("Umbral de probabilidad RF Type II")
plt.ylabel("F1 macro")
plt.grid(alpha=0.25)
plt.show()


## 13) Experimento C: features que aportan y candidatas a relleno

**Pregunta:** se revisa que variables utiliza realmente RF3.

La importancia de Random Forest es una guia descriptiva. Una feature con importancia menor a `0.5%` se marca como candidata de relleno, pero solo debe eliminarse si una nueva evaluacion confirma que no perjudica F1 macro.


In [ ]:

# Analisis: revisar que variables utiliza el Random Forest final de Type III/IV.
ImportanciasT34 = ResultadoModelo["feature_importance_tipo_iii_iv"].copy()
ImportanciasT34["importance_acumulada"] = ImportanciasT34["importance"].cumsum()
# Un valor menor a 0.5% se marca como candidato de relleno; no implica eliminarlo automaticamente.
UMBRAL_RELLENO = 0.005
FeaturesRelleno = ImportanciasT34[ImportanciasT34["importance"] < UMBRAL_RELLENO].copy()

print("Top 20 variables del Random Forest Type III/IV:")
display(ImportanciasT34.head(20).round(6))
print("Features candidatas a relleno (importancia menor a 0.5%):", len(FeaturesRelleno))
display(FeaturesRelleno[["feature", "importance"]].head(20).round(6))

# Esta grafica sigue el estilo de clase: una barra permite identificar aportes dominantes.
TopImportancias = ImportanciasT34.head(20).sort_values("importance")
plt.figure(figsize=(10, 7))
plt.barh(TopImportancias["feature"], TopImportancias["importance"], color="steelblue")
plt.axvline(UMBRAL_RELLENO, color="red", linestyle="--", label="Umbral de relleno")
plt.title("Importancia de features - RF Type III/IV")
plt.xlabel("Importancia")
plt.legend()
plt.tight_layout()
plt.show()


## 14) Visualizacion del espacio de pares con PCA

**Que se hace:** proyectar el vector enriquecido a dos componentes y colorear cada par por su clase real.

**Como leerlo:** puntos cercanos tienen perfiles similares en las features; las `x` negras son errores del RF3. PC1 y PC2 son combinaciones de variables, no una caracteristica individual.


In [ ]:
# Bloque: PCA del espacio de pares y zoom Type III/IV.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Espacio comun para visualizacion; el modelo jerarquico conserva sus features de entrenamiento.
DatosTest = DatosModelo[DatosModelo["split"] == "test"].copy()
XVisualTest = construir_features_tipo_iii_iv(
    DatosTest,
    min_match_len=MIN_MATCH_LEN_BAKER,
    ast_variant=AST_VARIANT_OFICIAL,
)

PCAVisual = PCA(n_components=2, random_state=SEED)
PuntosTest = PCAVisual.fit_transform(StandardScaler().fit_transform(XVisualTest))
DatosVisual = DatosTest[["filename", "clone_type"]].copy()
DatosVisual["prediccion"] = ResultadoModelo["pred_test"].reindex(DatosTest.index)
DatosVisual["pc1"] = PuntosTest[:, 0]
DatosVisual["pc2"] = PuntosTest[:, 1]
DatosVisual["correcto"] = DatosVisual["clone_type"] == DatosVisual["prediccion"]

ColoresClase = {
    "type_I": "#16697a",
    "type_II": "#f4a261",
    "type_III": "#2a9d8f",
    "type_IV": "#e76f51",
}
fig, Ejes = plt.subplots(1, 2, figsize=(15, 6))

# Recorre los elementos para acumular la salida correspondiente de este paso.
for Clase in ETIQUETAS_MODELO:
    PuntosClase = DatosVisual[DatosVisual["clone_type"] == Clase]
    Ejes[0].scatter(
        PuntosClase["pc1"], PuntosClase["pc2"], s=25, alpha=0.67,
        color=ColoresClase[Clase], label=Clase,
    )
Ejes[0].set_title(
    f"PCA TEST - cuatro clases ({PCAVisual.explained_variance_ratio_.sum():.1%} varianza)"
)
Ejes[0].set_xlabel("Componente principal 1")
Ejes[0].set_ylabel("Componente principal 2")
Ejes[0].legend()
Ejes[0].grid(alpha=0.2)

MascaraT34 = DatosVisual["clone_type"].isin(["type_III", "type_IV"])
VistaT34 = DatosVisual.loc[MascaraT34].copy()
PCAVisualT34 = PCA(n_components=2, random_state=SEED)
PuntosT34 = PCAVisualT34.fit_transform(
    StandardScaler().fit_transform(XVisualTest.loc[MascaraT34])
)
VistaT34["pc1"] = PuntosT34[:, 0]
VistaT34["pc2"] = PuntosT34[:, 1]

# Recorre los elementos para acumular la salida correspondiente de este paso.
for Clase in ["type_III", "type_IV"]:
    PuntosClase = VistaT34[VistaT34["clone_type"] == Clase]
    Ejes[1].scatter(
        PuntosClase["pc1"], PuntosClase["pc2"], s=27, alpha=0.7,
        color=ColoresClase[Clase], label=Clase,
    )
ErroresT34 = VistaT34[~VistaT34["correcto"]]
Ejes[1].scatter(
    ErroresT34["pc1"], ErroresT34["pc2"], s=62, marker="x",
    linewidths=1.5, color="#111111", label="Error del RF",
)
Ejes[1].set_title(
    f"Zoom type_III/type_IV ({PCAVisualT34.explained_variance_ratio_.sum():.1%} varianza)"
)
Ejes[1].set_xlabel("Componente principal 1")
Ejes[1].set_ylabel("Componente principal 2")
Ejes[1].legend()
Ejes[1].grid(alpha=0.2)

plt.tight_layout()
plt.show()

print("Varianza explicada PCA cuatro clases:", round(float(PCAVisual.explained_variance_ratio_.sum()), 4))
print("Varianza explicada PCA type_III/type_IV:", round(float(PCAVisualT34.explained_variance_ratio_.sum()), 4))
print("Errores type_III/type_IV en TEST:", len(ErroresT34))
display(ErroresT34[["filename", "clone_type", "prediccion"]].head(10))


## 15) Interpretacion de componentes y pares importantes

**Que se hace:** mostrar las cargas PCA y graficar pares de features dominantes para Type III/IV.

**Diferencia importante:** las cargas explican la forma del dibujo PCA; la importancia del Random Forest explica decisiones predictivas.


In [ ]:
# Bloque: cargas PCA y graficas de pares dominantes.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
# Funcion: Calcula cuanto participa cada feature en PC1 y PC2.
def construir_tabla_cargas_pca(modelo_pca: PCA, columnas: list[str]) -> pd.DataFrame:
    cargas = pd.DataFrame(
        modelo_pca.components_.T,
        index=columnas,
        columns=["PC1", "PC2"],
    )
    cargas["peso_total_2d"] = np.sqrt(cargas["PC1"] ** 2 + cargas["PC2"] ** 2)
    return cargas.sort_values("peso_total_2d", ascending=False)


CargasPCA = construir_tabla_cargas_pca(PCAVisual, list(XVisualTest.columns))
CargasPCAT34 = construir_tabla_cargas_pca(PCAVisualT34, list(XVisualTest.columns))
Top10PCA = CargasPCA.head(10)
Top10PCAT34 = CargasPCAT34.head(10)

print("Top 10 features que mas forman los ejes PCA - cuatro clases:")
display(Top10PCA.round(4))
print("Top 10 features que mas forman los ejes PCA - zoom type_III/type_IV:")
display(Top10PCAT34.round(4))

fig, EjesCargas = plt.subplots(1, 2, figsize=(18, 7))
for Eje, Tabla, Titulo in [
    (EjesCargas[0], Top10PCA, "Top 10 cargas PCA - cuatro clases"),
    (EjesCargas[1], Top10PCAT34, "Top 10 cargas PCA - type_III/type_IV"),
]:
    TablaPlot = Tabla.sort_values("peso_total_2d")
    Posiciones = np.arange(len(TablaPlot))
    Eje.barh(Posiciones - 0.19, TablaPlot["PC1"], height=0.36, label="PC1", color="#16697a")
    Eje.barh(Posiciones + 0.19, TablaPlot["PC2"], height=0.36, label="PC2", color="#e76f51")
    Eje.set_yticks(Posiciones)
    Eje.set_yticklabels(TablaPlot.index)
    Eje.axvline(0, color="#333333", linewidth=0.8)
    Eje.set_xlabel("Carga PCA firmada")
    Eje.set_title(Titulo)
    Eje.legend()
    Eje.grid(axis="x", alpha=0.2)
plt.tight_layout()
plt.show()

FeaturesTop10T34 = Top10PCAT34.index.tolist()
ParesFeaturesTop10T34 = [
    (FeaturesTop10T34[i], FeaturesTop10T34[i + 1])
    for i in range(0, len(FeaturesTop10T34) - 1, 2)
]
DatosFeaturesT34 = XVisualTest.loc[MascaraT34].copy()
DatosFeaturesT34["clone_type"] = VistaT34["clone_type"]
DatosFeaturesT34["correcto"] = VistaT34["correcto"]

fig, EjesPares = plt.subplots(2, 3, figsize=(19, 11))
EjesPares = EjesPares.ravel()
# Recorre los elementos para acumular la salida correspondiente de este paso.
for Eje, (FeatureX, FeatureY) in zip(EjesPares, ParesFeaturesTop10T34):
    # Recorre los elementos para acumular la salida correspondiente de este paso.
    for Clase in ["type_III", "type_IV"]:
        Subset = DatosFeaturesT34[DatosFeaturesT34["clone_type"] == Clase]
        Eje.scatter(
            Subset[FeatureX], Subset[FeatureY], s=25, alpha=0.65,
            color=ColoresClase[Clase], label=Clase,
        )
    ErroresPar = DatosFeaturesT34[~DatosFeaturesT34["correcto"]]
    Eje.scatter(
        ErroresPar[FeatureX], ErroresPar[FeatureY], s=58, marker="x",
        linewidths=1.4, color="#111111", label="Error RF",
    )
    Eje.set_xlabel(FeatureX)
    Eje.set_ylabel(FeatureY)
    Eje.grid(alpha=0.2)
# Recorre los elementos para acumular la salida correspondiente de este paso.
for Eje in EjesPares[len(ParesFeaturesTop10T34):]:
    Eje.axis("off")
EjesPares[0].legend()
fig.suptitle("Pares de features dominantes del PCA para type_III/type_IV", y=1.01)
plt.tight_layout()
plt.show()


## 16) Resumen reproducible

Se conserva un diccionario final con metricas e hiperparametros para explicar el resultado sin escribir reportes externos.


In [ ]:
# Bloque: resumen del modelo en memoria.
# Las instrucciones siguientes conservan la logica funcional del modelo actual.
ResumenModelo = {
    "metricas_val": ResultadoModelo["metricas_val"],
    "metricas_test": ResultadoModelo["metricas_test"],
    "resumen_pred_val": ResultadoModelo["resumen_pred_val"],
    "resumen_pred_test": ResultadoModelo["resumen_pred_test"],
    "cfg_tipo_ii": ResultadoModelo["cfg_tipo_ii"].__dict__,
    "cfg_tipo_iii_iv": ResultadoModelo["cfg_tipo_iii_iv"].__dict__,
    "umbral_tipo_i": ResultadoModelo["umbral_tipo_i"],
    "umbral_prob_tipo_ii": ResultadoModelo["umbral_prob_tipo_ii"],
    "num_features_tipo_ii": ResultadoModelo["num_features_tipo_ii"],
    "num_features_tipo_iii_iv": ResultadoModelo["num_features_tipo_iii_iv"],
    "f1_val_tipo_ii": ResultadoModelo["f1_val_tipo_ii"],
    "f1_val_tipo_iii_iv": ResultadoModelo["f1_val_tipo_iii_iv"],
}

print("Resumen del modelo disponible en variable: ResumenModelo")
print("No se escribieron archivos externos.")
print("\nJSON (preview):")
print(json.dumps(ResumenModelo, ensure_ascii=False, indent=2)[:1800] + "\n...")
